# BRIDGE-JUPYTER-01 — PostgreSQL magics solution

## Goal

Demonstrate one credential-safe, read-only JupySQL workflow against the disposable course PostgreSQL database, including bounded results, named value parameters, pandas conversion, connection cleanup, and an explicit transaction example.

**Level:** Intermediate/advanced  
**Stable lesson ID:** `bridge-jupyter-01`  
**Prerequisites:** Python Day 18, SQL Day 15, Bridge Day 3, and a reset disposable course database.

`%sql` is a line magic for a short statement or result assignment. `%%sql` is a cell magic whose body is multi-line SQL.

## Setup

Set `DS60_DATABASE_URL` in the shell that launches VS Code or Jupyter. The notebook reads it once, never displays it, and rejects any database other than the disposable `advanced_sql_training` target.

In [ ]:
# ruff: noqa: E501 -- JupySQL line magics stay on one physical line.
%load_ext sql

In [ ]:
import os

from sqlalchemy import create_engine, text
from sqlalchemy.engine import make_url

raw_database_url = os.environ.get("DS60_DATABASE_URL", "").strip()
if not raw_database_url:
    raise RuntimeError(
        "Set DS60_DATABASE_URL in the shell that starts this notebook, then restart the kernel."
    )

course_url = make_url(raw_database_url)
if course_url.get_backend_name() not in {"postgres", "postgresql"}:
    raise RuntimeError("DS60_DATABASE_URL must select PostgreSQL.")
if course_url.database != "advanced_sql_training":
    raise RuntimeError("This lesson is restricted to the disposable course database.")

psycopg_url = course_url.set(drivername="postgresql+psycopg")
engine = create_engine(psycopg_url, pool_pre_ping=True)
assert engine.url.drivername == "postgresql+psycopg"

In [ ]:
%config SqlMagic.displaycon = False
%config SqlMagic.autolimit = 200
%config SqlMagic.displaylimit = 25
%config SqlMagic.autopandas = False
%config SqlMagic.named_parameters = "enabled"

In [ ]:
%sql engine --alias ds60-course

In [ ]:
%sql --connections

## Steps

### 1. Run bounded read-only SQL

The line diagnostic is short. The customer query is easier to review as a cell magic.

In [ ]:
%sql SELECT current_database() AS database_name, current_user AS database_user

In [ ]:
%%sql
SELECT customer_id, full_name, country, segment
FROM training.customers
ORDER BY customer_id
LIMIT 5;

### 2. Bind values with `:name`

The status and amount remain Python data. Neither value is rendered into SQL source.

In [ ]:
order_status = "paid"
minimum_total = 250

In [ ]:
orders_result = %sql SELECT order_id, customer_id, total_amount FROM training.orders WHERE status = :order_status AND total_amount >= :minimum_total ORDER BY total_amount DESC, order_id LIMIT 20

In [ ]:
orders_frame = orders_result.DataFrame()
assert list(orders_frame.columns) == ["order_id", "customer_id", "total_amount"]
assert len(orders_frame) <= 20
orders_frame.head()

### 3. Compare explicit conversion with `autopandas`

`autopandas=True` returns a DataFrame directly. The SQL still has its own `LIMIT` because `displaylimit` affects presentation, not fetched row count, and pandas display rules apply in this mode.

In [ ]:
%config SqlMagic.autopandas = True
customer_frame = %sql SELECT customer_id, full_name, country FROM training.customers ORDER BY customer_id LIMIT 10
%config SqlMagic.autopandas = False

In [ ]:
assert customer_frame.shape[0] <= 10
assert {"customer_id", "full_name", "country"}.issubset(customer_frame.columns)

### 4. Keep code generation out of the value path

Jinja `{{value}}` renders SQL code before execution and therefore requires trusted, reviewed input. Named `:value` syntax uses the parameter boundary and is the correct choice for data values. Neither form safely binds a table or column name. Keep notebook identifiers static; application code can combine an allowlist with `psycopg.sql.Identifier` when dynamism is truly required.

### 5. Make transaction ownership visible

JupySQL autocommit defaults to true. This lesson performs only reads. The SQLAlchemy block below demonstrates an explicit transaction context without changing data.

In [ ]:
with engine.begin() as connection:
    observed_database = connection.scalar(text("SELECT current_database()"))

assert observed_database == "advanced_sql_training"

### 6. Move reusable effects into Psycopg code

Magics are excellent for bounded exploration. Reusable typed queries, multiple writes, COPY, streaming, retry classification, pooling, async work, cancellation, structured logging, and unit-test seams belong in application modules using Psycopg or a deliberately chosen SQLAlchemy layer.

## Exercise solutions

These walkthroughs map one-for-one to the answer-free learner artifact and
companion guide. The executable reference is `bridge/professional/solutions/bridge_jupyter_01_postgresql_magics_solution.ipynb`.

**Shared failure rule:** Notebook output, connection displays, Jinja rendering, or unbounded result materialization can leak secrets or turn exploration into unsafe application behavior.

### Exercise 1 — Magic selection

**Prompt:** Run the line diagnostic and multi-line training query; explain why one `%sql` form
is easier to review for each statement.

**Approach:** Use line magic for one compact diagnostic or assignment and cell magic for
formatted multi-line SQL. Both use the same bound engine and require the same value-binding
discipline.

**Why:** Choose from statement shape and reviewability, not from different security semantics.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 2 — Security testing

**Prompt:** Bind an injection-shaped string to a harmless read-only comparison, verify it
behaves as data, then remove the value.

**Approach:** Assign the sentinel in Python, reference it through a named parameter, and query a
bounded comparison. The result should reflect literal data semantics rather than altering the
SQL structure.

**Why:** Use `:name`; never paste the sentinel into SQL or notebook output.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 3 — SQL practice

**Prompt:** Query US customers whose lifetime order total meets a Python threshold; bind country
and threshold, order deterministically, and limit to 10.

**Approach:** Use static schema/table names, `:exercise_country` and `:exercise_minimum_total`,
`COALESCE(sum(...), 0)`, `HAVING`, and final `ORDER BY total DESC, customer_id LIMIT 10`.

**Why:** Aggregate after a left join, apply the threshold after grouping, and add customer ID as
tie-breaker.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 4 — DataFrame boundary

**Prompt:** Convert the assigned result with `.DataFrame()` and assert the expected columns in
order.

**Approach:** Call `.DataFrame()` once, compare the exact column list with the query projection,
and assert the row count is bounded by 10 before using the frame.

**Why:** Treat conversion as an explicit boundary and validate shape before analysis.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 5 — Configuration

**Prompt:** Repeat one small query with `autopandas=True`, identify the changed return type,
then restore it to false.

**Approach:** Enable the option in a live-tagged cell, verify a DataFrame is returned directly,
and reset it in the same teaching section so later cells retain explicit result conversion.

**Why:** Notebook-global magic configuration is hidden state unless restored visibly.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 6 — Memory reasoning

**Prompt:** Explain why `displaylimit=25` does not bound memory and identify the setting/query
clause that does.

**Approach:** `displaylimit` truncates presentation only. Use SQL `LIMIT` and JupySQL
`autolimit` as fetch guards, while recognizing that an aggregate or expensive query can still do
substantial server work.

**Why:** Rendering fewer rows is different from fetching fewer rows.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 7 — Identifier boundary

**Prompt:** Explain why `FROM :table_name` is not identifier binding and how a Psycopg
application handles a validated dynamic identifier.

**Approach:** Keep notebook identifiers static. Application code first allowlists the requested
object, then composes it with `psycopg.sql.Identifier`; it does not pass a table name as `%s` or
`:name`.

**Why:** Bound parameters represent data values, never SQL grammar.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 8 — Architecture decision

**Prompt:** Write a decision note choosing interactive exploration or Psycopg application code
using at least three criteria.

**Approach:** Keep bounded one-off read exploration in the notebook; move reusable,
write-capable, dynamically composed, scheduled, or heavily tested behavior into typed
application code.

**Why:** Consider reuse, transaction ownership, tests, dynamic structure, scale, and operational
observability.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 9 — Cleanup

**Prompt:** List active aliases, close `ds60-course`, dispose the engine, and verify no
connection literal or output remains saved.

**Approach:** Inspect `%sql --connections`, close the named alias, call `engine.dispose()`,
clear all outputs/execution counts, and scan notebook JSON for URL-shaped credential text.

**Why:** Both JupySQL alias state and SQLAlchemy pool state need explicit cleanup.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 10 — Setup review

**Prompt:** Trace how the environment URL becomes a SQLAlchemy engine without ever being
displayed and identify every validation step.

**Approach:** Read the variable with `os.environ`, parse with `make_url`, require PostgreSQL
plus the `psycopg` driver and disposable database, set `displaycon=False`, create the engine,
then bind the object rather than a literal URL.

**Why:** Validate scheme, driver, host/database target, and display settings before connecting.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 11 — Prediction

**Prompt:** Predict the difference between a `%sql` line assignment and a `%%sql` cell when both
return the same rows.

**Approach:** A line magic can assign its result directly in Python on one line; a cell magic
owns the whole cell and emphasizes formatted SQL. Result semantics and parameter safety are
otherwise equivalent.

**Why:** Compare Python assignment syntax, multi-line readability, and result access.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 12 — Capacity

**Prompt:** Design a query/result-size check for a table with millions of rows and explain what
remains unbounded after `LIMIT 25`.

**Approach:** Add selective predicates and partition/date bounds, project only needed columns,
use deterministic `LIMIT`, and inspect a plan if authorized. `LIMIT` alone may still require a
large scan/sort and does not cap aggregate work.

**Why:** Bound output, scan scope, and server work separately.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 13 — Type binding

**Prompt:** Bind a `Decimal`, date, boolean, and list value in small read-only queries and
record their PostgreSQL result types.

**Approach:** Assign typed Python values, reference each with `:name`, and use bounded `SELECT
pg_typeof(...)` diagnostics. The adapter preserves value boundaries and exposes type mismatches
explicitly.

**Why:** Let SQLAlchemy/JupySQL adapt Python values; do not pre-render literals.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 14 — Transaction reasoning

**Prompt:** Use `engine.begin()` for a rollback-safe teaching write in a disposable temporary
scope, then explain why magics are not the transaction owner.

**Approach:** Application-level SQLAlchemy context owns commit/rollback and connection return.
JupySQL exploratory cells should remain read-only; any write lab needs a separate tagged,
opt-in, rollback-protected artifact.

**Why:** The checked-in notebook remains read-only; describe or run writes only in an explicitly
authorized disposable lab.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 15 — Jinja boundary

**Prompt:** Demonstrate conceptually why `{{value}}` is code generation rather than safe value
binding, including a harmless fixed example.

**Approach:** Use `:value` for data. Reserve Jinja for trusted, reviewed structure when
unavoidable; never pass user text through it and never describe rendering as equivalent to
driver binding.

**Why:** Rendered text becomes SQL before the driver sees parameters.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 16 — Notebook hygiene

**Prompt:** Validate nbformat, stable IDs, kernel metadata, live/static tags, empty outputs, and
the absence of package-install magics, shell installs, URLs, and destructive SQL.

**Approach:** Run `nbformat.validate` plus repository validators/tests, require the `ds60sqlpy`
kernel and unique IDs, clear every code output/count, and scan sources for forbidden
installation/credential/write patterns.

**Why:** Inspect the serialized artifact, not only the visible notebook UI.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 17 — Offline review

**Prompt:** Explain how a learner without a running PostgreSQL server can still review this
module and which claims remain unexecuted.

**Approach:** They can read all prose/SQL, validate notebook JSON/tags/security, and run
non-live Python cells where dependencies exist. Query results, adapter behavior, and cleanup
against PostgreSQL remain explicitly unverified until the opt-in live path runs.

**Why:** Separate structural/offline evidence from live query evidence.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.

### Exercise 18 — Handoff

**Prompt:** Extract the capstone query into a Psycopg function design with typed inputs, a small
cursor Protocol, and a fake-backed test.

**Approach:** Keep static SQL with `%s` placeholders, type country/threshold inputs, inject a
cursor Protocol, map bounded rows explicitly, and test query/parameter separation before an
optional live integration test.

**Why:** Carry over SQL and value semantics while changing the ownership/testing surface.

**Evidence:** Assert deterministic outputs plus the exact calls that did
and did not cross the boundary. Keep live/network evidence opt-in,
credential-free, bounded, and separately labeled.


## Checks

The completed aggregate binds country and threshold as data, preserves static identifiers, orders deterministically, and caps output.

In [ ]:
exercise_country = "US"
exercise_minimum_total = 500

In [ ]:
customer_totals = %sql SELECT c.customer_id, c.full_name, COALESCE(sum(o.total_amount), 0) AS lifetime_total FROM training.customers AS c LEFT JOIN training.orders AS o USING (customer_id) WHERE c.country = :exercise_country GROUP BY c.customer_id, c.full_name HAVING COALESCE(sum(o.total_amount), 0) >= :exercise_minimum_total ORDER BY lifetime_total DESC, c.customer_id LIMIT 10

In [ ]:
exercise_frame = customer_totals.DataFrame()
assert list(exercise_frame.columns) == ["customer_id", "full_name", "lifetime_total"]
assert len(exercise_frame) <= 10
exercise_frame

The safety check is behavioral: no connection literal or saved output exists; the result is bounded; the two inputs are named parameters; identifiers are fixed course objects; the query is read-only; and the connection is closed explicitly.

In [ ]:
%sql --close ds60-course

In [ ]:
engine.dispose()

## Next Steps

Return to Bridge Days 3–5 to package exploratory SQL behind typed Psycopg functions and fake-backed tests. Continue to BRIDGE-OPS-01 to connect migration checks, request IDs, retry policy, health/readiness, metrics, and forward-fix versus rollback evidence.